# Análisis de entrenamiento de BTC/USDT

Este cuaderno usa solo entrenamiento para explorar y justificar el estado. No muestra el periodo final. La instantánea del CSV y sus metadatos están incluidos.

In [ ]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "configs/default.json").exists(), "Abre el cuaderno desde el proyecto"
os.chdir(root)
sys.path.insert(0, str(root))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/rl_inversion_matplotlib")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data import prepare, FEATURES
cfg = json.loads(Path("configs/default.json").read_text())
df, x, splits, mu, sigma = prepare(cfg["data"], cfg)
train = df.iloc[:splits["train"][1]+1].copy()
print("EDA restringido a entrenamiento:", train.timestamp.iloc[0], "a", train.timestamp.iloc[-1])

## Calidad y descripción

Revisar orden, duplicados y frecuencia evita simular recorridos imposibles. El timestamp etiqueta la vela diaria UTC; su cierre se conoce al terminar el día.

In [ ]:
assert train.timestamp.is_monotonic_increasing
assert not train.timestamp.duplicated().any()
assert (train.close > 0).all()
display(train[["close", "volume"] + FEATURES].describe())
print("Saltos de fechas:", train.timestamp.diff().value_counts().to_dict())

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].plot(train.timestamp, train.close)
ax[0, 0].set(title="Precio de BTC/USDT", ylabel="USDT")
ax[0, 1].hist(train.return_1, bins=55)
ax[0, 1].set(title="Retornos diarios", xlabel="Retorno")
ax[1, 0].plot(train.timestamp, train.volatility_20)
ax[1, 0].set(title="Volatilidad diaria móvil de 20 días")
asset_dd = train.close / train.close.cummax() - 1
ax[1, 1].plot(train.timestamp, asset_dd)
ax[1, 1].set(title="Drawdown del activo")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print("Mayor caída diaria:", train.return_1.min())
print("Mayor subida diaria:", train.return_1.max())
print("Drawdown máximo del precio en entrenamiento:", asset_dd.min())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(train.timestamp, train.volume)
axes[0].set(ylabel="BTC", title="Volumen")
axes[1].plot(train.timestamp, train.momentum_5, label="5 días")
axes[1].plot(train.timestamp, train.momentum_20, label="20 días")
axes[1].legend()
axes[1].set(title="Momentum")
axes[2].plot(train.timestamp, train.gap_sma20)
axes[2].set(title="Separación relativa frente a la media de 20 días")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
high = train.nlargest(5, "volatility_20")[["timestamp", "volatility_20", "return_1"]]
low = train.nsmallest(5, "volatility_20")[["timestamp", "volatility_20", "return_1"]]
display(high)
display(low)
display(train[FEATURES].corr())

## Interpretación

La caída diaria más grande y el drawdown del activo muestran por qué el retorno acumulado no basta para evaluar al agente. La variación de volatilidad justifica incluir una medida móvil en el estado. Los retornos de 5 y 20 días describen cambios de tendencia sin depender de la cotización absoluta. La correlación entre indicadores permite detectar redundancia, aunque en este experimento se conserva la representación fijada antes de prueba.

El volumen se inspecciona y se excluye del estado para contener su dimensión. Los resultados no justifican hacer ajustes con el periodo final. La recompensa alternativa penaliza pérdidas netas diarias; su utilidad se decide en validación.

## Auditoría de causalidad

In [ ]:
from src.data import features, load_prices
raw = load_prices(cfg["data"])
changed = raw.copy()
cut = len(raw) - 50
changed.loc[cut:, "close"] *= 2
before = features(raw)
after = features(changed)
mask = before.timestamp < raw.timestamp.iloc[cut]
np.testing.assert_allclose(before.loc[mask, FEATURES], after.loc[mask, FEATURES])
print("Cambiar el futuro no modifica indicadores pasados.")